## Text Embedder Test
Day 5 작업 중 Text Embedder 모듈 기능 테스트

**IMPORTANT: 사전 준비 단계**
```bash
# 1. FastAPI 서버 실행 (별도 터미널)
uvicorn src.app.api.main:app --reload

# 2. .env 파일에 OPENAI_API_KEY 설정 확인
```

#### 테스트 대상 모듈
- `src/app/processors/embedder.py` - TextEmbedder 클래스

#### 테스트 항목
1. TextEmbedder 초기화
2. 토큰 카운팅 및 텍스트 자르기
3. 단일 텍스트 임베딩 생성
4. 캐싱 시스템 동작 확인
5. 배치 임베딩 처리
6. 아티클 임베딩 (제목+요약+본문)
7. 임베딩 유사도 계산
8. 에러 처리 및 재시도 로직

In [1]:
import sys
import asyncio
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 루트를 Python 경로에 추가
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# 환경 변수 로드
load_dotenv(project_root / ".env")

from src.app.processors.embedder import TextEmbedder, get_embedder
from src.app.core.config import settings

print("✓ Setup complete")
print(f"OpenAI Model: {settings.OPENAI_EMBEDDING_MODEL}")
print(f"Vector Size: {settings.QDRANT_VECTOR_SIZE}")

✓ Setup complete
OpenAI Model: text-embedding-3-small
Vector Size: 1536


### 1. TextEmbedder 초기화
TextEmbedder 인스턴스를 생성하고 설정을 확인

In [2]:
# TextEmbedder 인스턴스 생성
embedder = TextEmbedder(
    use_cache=True,
    max_retries=3,
    retry_wait_min=1,
    retry_wait_max=10,
)

print("TextEmbedder Configuration:")
print("=" * 60)
print(f"Model: {embedder.model}")
print(f"Max Tokens: {embedder.MAX_TOKENS}")
print(f"Cache Enabled: {embedder.use_cache}")
print(f"Max Retries: {embedder.max_retries}")
print(f"Embedding Dimension: {embedder.get_embedding_dimension()}")
print(f"Cache Size: {embedder.get_cache_size()}")

cache_stats = embedder.get_cache_stats()
print(f"\nCache Stats: {cache_stats}")

TextEmbedder Configuration:
Model: text-embedding-3-small
Max Tokens: 8191
Cache Enabled: True
Max Retries: 3
Embedding Dimension: 1536
Cache Size: 0

Cache Stats: {'size': 0, 'enabled': True, 'model': 'text-embedding-3-small'}


### 2. 토큰 카운팅 테스트

텍스트의 토큰 수를 계산하고, 토큰 제한을 초과하는 텍스트를 자름

In [3]:
# 테스트 텍스트들
test_texts = [
    "Attention Is All You Need",
    "Transformer 아키텍처를 제안하는 논문입니다.",
    "GPT-4 is a large multimodal model that can process both text and images.",
]

print("Token Counting Test:")
print("=" * 60)

for i, text in enumerate(test_texts, 1):
    token_count = embedder.count_tokens(text)
    print(f"[{i}] Text: {text}")
    print(f"    Tokens: {token_count}")
    print(f"    Characters: {len(text)}")
    print()

Token Counting Test:
[1] Text: Attention Is All You Need
    Tokens: 5
    Characters: 25

[2] Text: Transformer 아키텍처를 제안하는 논문입니다.
    Tokens: 19
    Characters: 29

[3] Text: GPT-4 is a large multimodal model that can process both text and images.
    Tokens: 18
    Characters: 72



In [4]:
# 긴 텍스트 자르기 테스트
print("Text Truncation Test:")
print("=" * 60)

# 매우 긴 텍스트 생성 (토큰 제한 초과)
long_text = "AI research and development " * 5000
token_count_before = embedder.count_tokens(long_text)

print(f"Original text:")
print(f"  Characters: {len(long_text)}")
print(f"  Tokens: {token_count_before}")
print(f"  Exceeds limit: {token_count_before > embedder.MAX_TOKENS}")

# 1000 토큰으로 자르기
truncated = embedder.truncate_text(long_text, max_tokens=1000)
token_count_after = embedder.count_tokens(truncated)

print(f"\nTruncated text (max 1000 tokens):")
print(f"  Characters: {len(truncated)}")
print(f"  Tokens: {token_count_after}")
print(f"  Within limit: {token_count_after <= 1000}")
print(f"  Preview: {truncated[:100]}...")

Text truncated from 20001 to 1000 tokens


Text Truncation Test:
Original text:
  Characters: 140000
  Tokens: 20001
  Exceeds limit: True

Truncated text (max 1000 tokens):
  Characters: 6999
  Tokens: 1000
  Within limit: True
  Preview: AI research and development AI research and development AI research and development AI research and ...


### 3. 단일 텍스트 임베딩 생성

하나의 텍스트를 임베딩으로 변환

In [5]:
print("Single Text Embedding Test:")
print("=" * 60)

text = "Attention Is All You Need"
print(f"Text: {text}")
print(f"Tokens: {embedder.count_tokens(text)}")
print("\nGenerating embedding...")

# 임베딩 생성 (비동기)
embedding = await embedder.embed(text)

print(f"\n✅ Embedding generated successfully!")
print(f"Dimension: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")
print(f"Data type: {type(embedding[0])}")
print(f"\nVector norm: {np.linalg.norm(embedding):.4f}")

Single Text Embedding Test:
Text: Attention Is All You Need
Tokens: 5

Generating embedding...

✅ Embedding generated successfully!
Dimension: 1536
First 10 values: [0.046672966331243515, 0.00821229349821806, -0.024438994005322456, 0.038446538150310516, -0.04568353295326233, -0.05450361967086792, -0.00843844935297966, 0.06021406129002571, -0.030898576602339745, 0.02120213396847248]
Data type: <class 'float'>

Vector norm: 1.0000


### 4. 캐싱 시스템 테스트

동일한 텍스트에 대해 캐시가 작동하는지 확인

In [6]:
print("Cache System Test:")
print("=" * 60)

# 캐시 초기화
embedder.clear_cache()
print(f"Cache cleared. Current size: {embedder.get_cache_size()}")

# 첫 번째 호출 - API 호출
text1 = "GPT-4 is a large language model"
print(f"\n[1st call] Text: {text1}")
print("Generating embedding (API call expected)...")

import time
start = time.time()
embedding1 = await embedder.embed(text1)
time1 = time.time() - start

print(f"✓ Time taken: {time1:.3f}s")
print(f"  Cache size: {embedder.get_cache_size()}")

# 두 번째 호출 - 캐시에서 가져오기
print(f"\n[2nd call] Same text: {text1}")
print("Getting embedding (cache hit expected)...")

start = time.time()
embedding2 = await embedder.embed(text1)
time2 = time.time() - start

print(f"✓ Time taken: {time2:.3f}s")
print(f"  Cache size: {embedder.get_cache_size()}")

# 결과 비교
print(f"\nCache Performance:")
print(f"  1st call time: {time1:.3f}s (API call)")
print(f"  2nd call time: {time2:.3f}s (cache hit)")
print(f"  Speedup: {time1/time2:.1f}x faster")
print(f"  Embeddings identical: {embedding1 == embedding2}")
print(f"\n✅ Cache working correctly!")

Cache System Test:
Cache cleared. Current size: 0

[1st call] Text: GPT-4 is a large language model
Generating embedding (API call expected)...
✓ Time taken: 1.067s
  Cache size: 1

[2nd call] Same text: GPT-4 is a large language model
Getting embedding (cache hit expected)...
✓ Time taken: 0.000s
  Cache size: 1

Cache Performance:
  1st call time: 1.067s (API call)
  2nd call time: 0.000s (cache hit)
  Speedup: 27118.3x faster
  Embeddings identical: True

✅ Cache working correctly!


In [7]:
# 캐시 키 생성 테스트
print("\nCache Key Generation Test:")
print("=" * 60)

texts = [
    "Attention Is All You Need",
    "Attention Is All You Need",  # 동일
    "Attention Is All You Need.",  # 마침표 추가 (다름)
    "attention is all you need",  # 소문자 (다름)
]

print("Text -> Cache Key (SHA-256 hash):")
for i, text in enumerate(texts, 1):
    cache_key = embedder._get_cache_key(text)
    print(f"[{i}] '{text}'")
    print(f"    {cache_key}")
    print()

# 동일한 텍스트는 같은 키를 생성
key1 = embedder._get_cache_key(texts[0])
key2 = embedder._get_cache_key(texts[1])
key3 = embedder._get_cache_key(texts[2])

print(f"Key comparison:")
print(f"  Text 1 == Text 2: {key1 == key2} (expected: True)")
print(f"  Text 1 == Text 3: {key1 == key3} (expected: False)")


Cache Key Generation Test:
Text -> Cache Key (SHA-256 hash):
[1] 'Attention Is All You Need'
    ab8508f26511d617d8cd58cf7d21103895b9180b46b48d8822d4f0a560d3e2e0

[2] 'Attention Is All You Need'
    ab8508f26511d617d8cd58cf7d21103895b9180b46b48d8822d4f0a560d3e2e0

[3] 'Attention Is All You Need.'
    611cdc0378e24f5dd2e2cc0979d788369c8bc935898dac056f27030b69de601f

[4] 'attention is all you need'
    4a1c934d3d98f3cefede93ebb74d0c4c2bac7ac9cc4f467d253ef8f87a2d6aad

Key comparison:
  Text 1 == Text 2: True (expected: True)
  Text 1 == Text 3: False (expected: False)


### 5. 배치 임베딩 처리

여러 텍스트를 한 번에 처리

In [8]:
print("Batch Embedding Test:")
print("=" * 60)

# 테스트 텍스트들
texts = [
    "Transformer architecture for NLP",
    "BERT: Pre-training of Deep Bidirectional Transformers",
    "GPT-4 Technical Report",
    "Attention mechanism in neural networks",
    "Reinforcement learning for robotics",
]

print(f"Processing {len(texts)} texts in batch...")
print(f"Batch size: 3")
print()

# 배치 임베딩 생성
start = time.time()
embeddings = await embedder.batch_embed(texts, batch_size=3, truncate=True)
elapsed = time.time() - start

print(f"\n✅ Batch embedding completed!")
print(f"Total embeddings: {len(embeddings)}")
print(f"Time taken: {elapsed:.2f}s")
print(f"Avg time per text: {elapsed/len(texts):.2f}s")

# 각 임베딩 확인
print(f"\nEmbedding Details:")
for i, (text, emb) in enumerate(zip(texts, embeddings), 1):
    tokens = embedder.count_tokens(text)
    print(f"[{i}] {text[:50]}...")
    print(f"    Tokens: {tokens}, Dimension: {len(emb)}, First 3: {emb[:3]}")

Batch Embedding Test:
Processing 5 texts in batch...
Batch size: 3


✅ Batch embedding completed!
Total embeddings: 5
Time taken: 3.20s
Avg time per text: 0.64s

Embedding Details:
[1] Transformer architecture for NLP...
    Tokens: 5, Dimension: 1536, First 3: [-0.05015554651618004, -0.015412251465022564, 0.045548148453235626]
[2] BERT: Pre-training of Deep Bidirectional Transform...
    Tokens: 9, Dimension: 1536, First 3: [-0.013022784143686295, -0.060584791004657745, 0.005793359130620956]
[3] GPT-4 Technical Report...
    Tokens: 6, Dimension: 1536, First 3: [-0.01882864162325859, 0.031000826507806778, 0.05904919654130936]
[4] Attention mechanism in neural networks...
    Tokens: 5, Dimension: 1536, First 3: [-0.0012657159240916371, 0.01806890033185482, -0.013505378738045692]
[5] Reinforcement learning for robotics...
    Tokens: 6, Dimension: 1536, First 3: [0.02214827574789524, -0.03853132203221321, 0.016505472362041473]


### 6. 아티클 임베딩 테스트

제목, 요약, 본문을 결합하여 아티클 임베딩을 생성

In [9]:
print("Article Embedding Test:")
print("=" * 60)

# 테스트 아티클
article = {
    "title": "Attention Is All You Need",
    "content": """
    The dominant sequence transduction models are based on complex recurrent or
    convolutional neural networks in an encoder-decoder configuration. The best
    performing models also connect the encoder and decoder through an attention
    mechanism. We propose a new simple network architecture, the Transformer,
    based solely on attention mechanisms, dispensing with recurrence and convolutions
    entirely. Experiments on two machine translation tasks show these models to be
    superior in quality while being more parallelizable and requiring significantly
    less time to train.
    """,
    "summary": "Transformer 아키텍처를 제안하는 논문으로, 순환 신경망 없이 attention만으로 구성됩니다.",
}

# 아티클 텍스트 준비
prepared_text = embedder.prepare_article_text(
    title=article["title"],
    content=article["content"],
    summary=article["summary"],
)

print(f"Article Title: {article['title']}")
print(f"\nPrepared Text:")
print(f"{prepared_text[:200]}...")
print(f"\nPrepared text stats:")
print(f"  Characters: {len(prepared_text)}")
print(f"  Tokens: {embedder.count_tokens(prepared_text)}")

# 아티클 임베딩 생성
print(f"\nGenerating article embedding...")
article_embedding = await embedder.embed_article(
    title=article["title"],
    content=article["content"],
    summary=article["summary"],
)

print(f"\n✅ Article embedding generated!")
print(f"Dimension: {len(article_embedding)}")
print(f"First 5 values: {article_embedding[:5]}")

Article Embedding Test:
Article Title: Attention Is All You Need

Prepared Text:
Title: Attention Is All You Need

Summary: Transformer 아키텍처를 제안하는 논문으로, 순환 신경망 없이 attention만으로 구성됩니다.

Content: 
    The dominant sequence transduction models are based on complex recurrent or
    con...

Prepared text stats:
  Characters: 712
  Tokens: 157

Generating article embedding...

✅ Article embedding generated!
Dimension: 1536
First 5 values: [0.02016959898173809, 0.003102353075519204, -0.03611471876502037, 0.0025900774635374546, 0.009602658450603485]


In [10]:
# 여러 아티클 배치 임베딩
print("\nMultiple Articles Batch Embedding Test:")
print("=" * 60)

articles = [
    {
        "title": "GPT-4 Technical Report",
        "content": "GPT-4 is a large-scale multimodal model that can process text and images.",
        "summary": "GPT-4는 텍스트와 이미지를 처리할 수 있는 대규모 멀티모달 모델입니다.",
    },
    {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers",
        "content": "BERT obtains new state-of-the-art results on eleven NLP tasks.",
        "summary": "BERT는 양방향 Transformer를 사용한 사전학습 모델입니다.",
    },
    {
        "title": "ResNet: Deep Residual Learning",
        "content": "Residual connections enable training of very deep neural networks.",
        "summary": "ResNet은 잔차 연결을 통해 매우 깊은 신경망 학습을 가능하게 합니다.",
    },
]

print(f"Processing {len(articles)} articles...")

# 배치 임베딩 생성
article_embeddings = await embedder.embed_articles_batch(articles, batch_size=2)

print(f"\n✅ {len(article_embeddings)} article embeddings generated!")
for i, (article, emb) in enumerate(zip(articles, article_embeddings), 1):
    print(f"[{i}] {article['title'][:50]}...")
    print(f"    Dimension: {len(emb)}")


Multiple Articles Batch Embedding Test:
Processing 3 articles...

✅ 3 article embeddings generated!
[1] GPT-4 Technical Report...
    Dimension: 1536
[2] BERT: Pre-training of Deep Bidirectional Transform...
    Dimension: 1536
[3] ResNet: Deep Residual Learning...
    Dimension: 1536


### 7. 임베딩 유사도 계산
생성된 임베딩들 간의 코사인 유사도를 계산

In [11]:
print("Embedding Similarity Test:")
print("=" * 60)

# 유사도 계산 함수
def cosine_similarity(a, b):
    """코사인 유사도 계산"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 테스트 텍스트들 (주제별로 그룹화)
test_texts = [
    "Transformer architecture for natural language processing",  # NLP
    "BERT and GPT models in NLP",  # NLP (유사)
    "Convolutional neural networks for image recognition",  # CV
    "ResNet and VGG in computer vision",  # CV (유사)
    "Reinforcement learning for game playing",  # RL
]

# 임베딩 생성
print("Generating embeddings for similarity test...")
test_embeddings = await embedder.batch_embed(test_texts, batch_size=5)

# 유사도 행렬 계산
print(f"\nCosine Similarity Matrix:")
print(f"{'':50s} " + " ".join([f"[{i+1}]" for i in range(len(test_texts))]))
print("-" * 80)

for i, (text_i, emb_i) in enumerate(zip(test_texts, test_embeddings)):
    row = f"[{i+1}] {text_i[:45]:45s} "
    for j, emb_j in enumerate(test_embeddings):
        if i == j:
            row += " 1.00"
        elif i < j:
            similarity = cosine_similarity(emb_i, emb_j)
            row += f" {similarity:.2f}"
        else:
            row += "     "
    print(row)

# 특정 쌍의 유사도 분석
print(f"\n\nSimilarity Analysis:")
print("-" * 80)

pairs = [
    (0, 1, "NLP topics (should be high)"),
    (2, 3, "CV topics (should be high)"),
    (0, 2, "NLP vs CV (should be lower)"),
    (1, 4, "NLP vs RL (should be lower)"),
]

for i, j, desc in pairs:
    similarity = cosine_similarity(test_embeddings[i], test_embeddings[j])
    print(f"[{i+1}] vs [{j+1}]: {similarity:.4f} - {desc}")
    print(f"  '{test_texts[i]}'")
    print(f"  '{test_texts[j]}'")
    print()

Embedding Similarity Test:
Generating embeddings for similarity test...

Cosine Similarity Matrix:
                                                   [1] [2] [3] [4] [5]
--------------------------------------------------------------------------------
[1] Transformer architecture for natural language  1.00 0.51 0.32 0.33 0.26
[2] BERT and GPT models in NLP                          1.00 0.24 0.37 0.24
[3] Convolutional neural networks for image recog            1.00 0.49 0.22
[4] ResNet and VGG in computer vision                             1.00 0.27
[5] Reinforcement learning for game playing                            1.00


Similarity Analysis:
--------------------------------------------------------------------------------
[1] vs [2]: 0.5066 - NLP topics (should be high)
  'Transformer architecture for natural language processing'
  'BERT and GPT models in NLP'

[3] vs [4]: 0.4897 - CV topics (should be high)
  'Convolutional neural networks for image recognition'
  'ResNet and VGG i

### 8. 에러 처리 테스트

잘못된 입력에 대한 에러 처리를 확인

In [12]:
print("Error Handling Test:")
print("=" * 60)

# 1. 빈 텍스트
print("[Test 1] Empty text:")
try:
    await embedder.embed("")
    print("  ❌ Should have raised ValueError")
except ValueError as e:
    print(f"  ✅ Caught expected error: {e}")

# 2. 공백만 있는 텍스트
print("\n[Test 2] Whitespace only text:")
try:
    await embedder.embed("   \n\t  ")
    print("  ❌ Should have raised ValueError")
except ValueError as e:
    print(f"  ✅ Caught expected error: {e}")

# 3. 배치 처리 시 일부 실패 (fail_on_error=False)
print("\n[Test 3] Batch with empty text (fail_on_error=False):")
mixed_texts = [
    "Valid text 1",
    "",  # 빈 텍스트 (에러)
    "Valid text 2",
]

try:
    batch_results = await embedder.batch_embed(mixed_texts, batch_size=3, fail_on_error=False)
    print(f"  ✅ Batch completed with errors handled")
    print(f"  Total results: {len(batch_results)}")
    for i, result in enumerate(batch_results, 1):
        is_zero_vector = all(v == 0.0 for v in result)
        print(f"    [{i}] Zero vector: {is_zero_vector} (dimension: {len(result)})")
except Exception as e:
    print(f"  ❌ Unexpected error: {e}")

# 4. 배치 처리 시 일부 실패 (fail_on_error=True)
print("\n[Test 4] Batch with empty text (fail_on_error=True):")
try:
    batch_results = await embedder.batch_embed(mixed_texts, batch_size=3, fail_on_error=True)
    print(f"  ❌ Should have raised exception")
except ValueError as e:
    print(f"  ✅ Caught expected error: {type(e).__name__}")

Error Handling Test:
[Test 1] Empty text:
  ✅ Caught expected error: Empty text provided for embedding

[Test 2] Whitespace only text:
  ✅ Caught expected error: Empty text provided for embedding

[Test 3] Batch with empty text (fail_on_error=False):


Error embedding text 1: Empty text provided for embedding
Error embedding text 1: Empty text provided for embedding


  ✅ Batch completed with errors handled
  Total results: 3
    [1] Zero vector: False (dimension: 1536)
    [2] Zero vector: True (dimension: 1536)
    [3] Zero vector: False (dimension: 1536)

[Test 4] Batch with empty text (fail_on_error=True):
  ✅ Caught expected error: ValueError


### 9. 싱글톤 패턴 테스트

get_embedder() 함수의 싱글톤 패턴을 검증

In [13]:
print("Singleton Pattern Test:")
print("=" * 60)

# 여러 번 호출
embedder1 = get_embedder()
embedder2 = get_embedder()
embedder3 = get_embedder()

# 같은 인스턴스인지 확인
print(f"embedder1 is embedder2: {embedder1 is embedder2}")
print(f"embedder2 is embedder3: {embedder2 is embedder3}")
print(f"embedder1 is embedder3: {embedder1 is embedder3}")

# 메모리 주소 확인
print(f"\nMemory addresses:")
print(f"  embedder1: {id(embedder1)}")
print(f"  embedder2: {id(embedder2)}")
print(f"  embedder3: {id(embedder3)}")

if embedder1 is embedder2 is embedder3:
    print(f"\n✅ Singleton pattern working correctly!")
    print(f"All get_embedder() calls return the same instance.")
else:
    print(f"\n❌ Singleton pattern not working!")

# 캐시 공유 확인
print(f"\nCache sharing test:")
embedder1.clear_cache()
print(f"embedder1 cache cleared")
print(f"embedder1 cache size: {embedder1.get_cache_size()}")
print(f"embedder2 cache size: {embedder2.get_cache_size()}")
print(f"embedder3 cache size: {embedder3.get_cache_size()}")

# embedder1로 임베딩 생성
await embedder1.embed("Test for cache sharing")
print(f"\nAfter embedder1.embed():")
print(f"embedder1 cache size: {embedder1.get_cache_size()}")
print(f"embedder2 cache size: {embedder2.get_cache_size()}")
print(f"embedder3 cache size: {embedder3.get_cache_size()}")

if embedder1.get_cache_size() == embedder2.get_cache_size() == embedder3.get_cache_size():
    print(f"\n✅ Cache is shared across all instances!")
else:
    print(f"\n❌ Cache is not shared!")

Singleton Pattern Test:
embedder1 is embedder2: True
embedder2 is embedder3: True
embedder1 is embedder3: True

Memory addresses:
  embedder1: 140347781342336
  embedder2: 140347781342336
  embedder3: 140347781342336

✅ Singleton pattern working correctly!
All get_embedder() calls return the same instance.

Cache sharing test:
embedder1 cache cleared
embedder1 cache size: 0
embedder2 cache size: 0
embedder3 cache size: 0

After embedder1.embed():
embedder1 cache size: 1
embedder2 cache size: 1
embedder3 cache size: 1

✅ Cache is shared across all instances!


### 10. 성능 벤치마크

임베딩 생성 성능을 측정

In [14]:
print("Performance Benchmark:")
print("=" * 60)

# 테스트 데이터 생성
benchmark_texts = [
    f"This is test article number {i} about AI and machine learning."
    for i in range(20)
]

# 캐시 비활성화 테스트
print("[1] Without cache:")
embedder_no_cache = TextEmbedder(use_cache=False)

start = time.time()
embeddings_no_cache = await embedder_no_cache.batch_embed(
    benchmark_texts[:10], batch_size=5
)
time_no_cache = time.time() - start

print(f"  Time: {time_no_cache:.2f}s")
print(f"  Texts processed: {len(embeddings_no_cache)}")
print(f"  Avg time per text: {time_no_cache/len(embeddings_no_cache):.2f}s")

# 캐시 활성화 테스트
print("\n[2] With cache (first run):")
embedder_with_cache = TextEmbedder(use_cache=True)

start = time.time()
embeddings_with_cache = await embedder_with_cache.batch_embed(
    benchmark_texts[:10], batch_size=5
)
time_with_cache_first = time.time() - start

print(f"  Time: {time_with_cache_first:.2f}s")
print(f"  Cache size: {embedder_with_cache.get_cache_size()}")

# 캐시 히트 테스트 (동일한 텍스트 재처리)
print("\n[3] With cache (second run, same texts):")

start = time.time()
embeddings_cached = await embedder_with_cache.batch_embed(
    benchmark_texts[:10], batch_size=5
)
time_cached = time.time() - start

print(f"  Time: {time_cached:.2f}s")
print(f"  Cache size: {embedder_with_cache.get_cache_size()}")

# 성능 비교
print(f"\nPerformance Comparison:")
print(f"  No cache:           {time_no_cache:.2f}s")
print(f"  With cache (1st):   {time_with_cache_first:.2f}s")
print(f"  With cache (2nd):   {time_cached:.2f}s")
print(f"  Speedup (cache hit): {time_no_cache/time_cached:.1f}x")

Performance Benchmark:
[1] Without cache:
  Time: 3.08s
  Texts processed: 10
  Avg time per text: 0.31s

[2] With cache (first run):
  Time: 1.69s
  Cache size: 10

[3] With cache (second run, same texts):
  Time: 0.50s
  Cache size: 10

Performance Comparison:
  No cache:           3.08s
  With cache (1st):   1.69s
  With cache (2nd):   0.50s
  Speedup (cache hit): 6.1x


### 11. 실전 사용 예시: 아티클 검색

임베딩을 사용한 시맨틱 검색을 시뮬레이션

In [15]:
print("Semantic Search Simulation:")
print("=" * 60)

# 데이터베이스에 저장된 아티클들 (시뮬레이션)
database_articles = [
    {
        "id": 1,
        "title": "Attention Is All You Need",
        "summary": "Transformer 아키텍처를 제안하는 논문입니다.",
    },
    {
        "id": 2,
        "title": "BERT: Pre-training of Deep Bidirectional Transformers",
        "summary": "양방향 Transformer 사전학습 모델입니다.",
    },
    {
        "id": 3,
        "title": "ResNet: Deep Residual Learning for Image Recognition",
        "summary": "잔차 연결을 통한 깊은 신경망 학습 방법입니다.",
    },
    {
        "id": 4,
        "title": "AlphaGo: Mastering the game of Go with deep learning",
        "summary": "강화학습을 통해 바둑 게임을 마스터한 AI입니다.",
    },
    {
        "id": 5,
        "title": "GPT-4 Technical Report",
        "summary": "대규모 언어 모델 GPT-4의 기술 보고서입니다.",
    },
]

# 아티클들의 임베딩 생성
print("Generating embeddings for database articles...")
db_texts = [f"{article['title']} {article['summary']}" for article in database_articles]
db_embeddings = await embedder.batch_embed(db_texts, batch_size=5)

# 검색 쿼리
queries = [
    "자연어 처리를 위한 Transformer 모델",
    "이미지 인식을 위한 딥러닝",
    "게임 AI와 강화학습",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    # 쿼리 임베딩 생성
    query_embedding = await embedder.embed(query)
    
    # 모든 아티클과의 유사도 계산
    similarities = [
        (i, cosine_similarity(query_embedding, db_emb))
        for i, db_emb in enumerate(db_embeddings)
    ]
    
    # 유사도 순으로 정렬
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # 상위 3개 결과 출력
    print(f"\nTop 3 Results:")
    for rank, (idx, similarity) in enumerate(similarities[:3], 1):
        article = database_articles[idx]
        print(f"\n[{rank}] Similarity: {similarity:.4f}")
        print(f"    ID: {article['id']}")
        print(f"    Title: {article['title']}")
        print(f"    Summary: {article['summary']}")

Semantic Search Simulation:
Generating embeddings for database articles...

Query: '자연어 처리를 위한 Transformer 모델'

Top 3 Results:

[1] Similarity: 0.4998
    ID: 2
    Title: BERT: Pre-training of Deep Bidirectional Transformers
    Summary: 양방향 Transformer 사전학습 모델입니다.

[2] Similarity: 0.4437
    ID: 1
    Title: Attention Is All You Need
    Summary: Transformer 아키텍처를 제안하는 논문입니다.

[3] Similarity: 0.4040
    ID: 5
    Title: GPT-4 Technical Report
    Summary: 대규모 언어 모델 GPT-4의 기술 보고서입니다.

Query: '이미지 인식을 위한 딥러닝'

Top 3 Results:

[1] Similarity: 0.3695
    ID: 3
    Title: ResNet: Deep Residual Learning for Image Recognition
    Summary: 잔차 연결을 통한 깊은 신경망 학습 방법입니다.

[2] Similarity: 0.3066
    ID: 2
    Title: BERT: Pre-training of Deep Bidirectional Transformers
    Summary: 양방향 Transformer 사전학습 모델입니다.

[3] Similarity: 0.3043
    ID: 4
    Title: AlphaGo: Mastering the game of Go with deep learning
    Summary: 강화학습을 통해 바둑 게임을 마스터한 AI입니다.

Query: '게임 AI와 강화학습'

Top 3 Results:

[1] Similarit

### 12. 전체 테스트 요약

In [16]:
print("\n" + "=" * 80)
print("✅ Text Embedder 테스트 완료")
print("=" * 80)
print("""
테스트 완료된 항목:
  1. ✓ TextEmbedder 초기화 및 설정 확인
  2. ✓ 토큰 카운팅 (count_tokens)
  3. ✓ 텍스트 자르기 (truncate_text)
  4. ✓ 단일 텍스트 임베딩 생성 (embed)
  5. ✓ 캐싱 시스템 (SHA-256 해시, 캐시 히트/미스)
  6. ✓ 배치 임베딩 처리 (batch_embed)
  7. ✓ 아티클 임베딩 (prepare_article_text, embed_article)
  8. ✓ 여러 아티클 배치 처리 (embed_articles_batch)
  9. ✓ 임베딩 유사도 계산 (코사인 유사도)
 10. ✓ 에러 처리 (빈 텍스트, fail_on_error 옵션)
 11. ✓ 싱글톤 패턴 검증 (get_embedder)
 12. ✓ 성능 벤치마크 (캐시 효과 측정)
 13. ✓ 시맨틱 검색 시뮬레이션

모든 테스트가 정상적으로 완료되었습니다! 🎉

주요 확인 사항:
- 임베딩 차원: 1536 (text-embedding-3-small)
- 최대 토큰: 8191
- 캐싱을 통한 성능 향상 확인
- 배치 처리로 여러 텍스트 동시 처리 가능
- 아티클 검색을 위한 시맨틱 유사도 계산 정상 작동
""")
print("=" * 80)


✅ Text Embedder 테스트 완료

테스트 완료된 항목:
  1. ✓ TextEmbedder 초기화 및 설정 확인
  2. ✓ 토큰 카운팅 (count_tokens)
  3. ✓ 텍스트 자르기 (truncate_text)
  4. ✓ 단일 텍스트 임베딩 생성 (embed)
  5. ✓ 캐싱 시스템 (SHA-256 해시, 캐시 히트/미스)
  6. ✓ 배치 임베딩 처리 (batch_embed)
  7. ✓ 아티클 임베딩 (prepare_article_text, embed_article)
  8. ✓ 여러 아티클 배치 처리 (embed_articles_batch)
  9. ✓ 임베딩 유사도 계산 (코사인 유사도)
 10. ✓ 에러 처리 (빈 텍스트, fail_on_error 옵션)
 11. ✓ 싱글톤 패턴 검증 (get_embedder)
 12. ✓ 성능 벤치마크 (캐시 효과 측정)
 13. ✓ 시맨틱 검색 시뮬레이션

모든 테스트가 정상적으로 완료되었습니다! 🎉

주요 확인 사항:
- 임베딩 차원: 1536 (text-embedding-3-small)
- 최대 토큰: 8191
- 캐싱을 통한 성능 향상 확인
- 배치 처리로 여러 텍스트 동시 처리 가능
- 아티클 검색을 위한 시맨틱 유사도 계산 정상 작동



### 13. 정리 (Cleanup)

테스트 종료 후 캐시를 정리

In [17]:
print("Cleanup:")
print("=" * 60)

# 캐시 정리
final_cache_size = embedder.get_cache_size()
print(f"Current cache size: {final_cache_size}")

embedder.clear_cache()
print(f"Cache cleared. New size: {embedder.get_cache_size()}")

print("\n✓ Cleanup complete")

Cleanup:
Current cache size: 25
Cache cleared. New size: 0

✓ Cleanup complete
